In [ ]:
import pandas as pd
import numpy as np
import iqplot
import glob

from plot_tools import *
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')


In [ ]:
assembly_metadata = pd.read_csv('assembly_metadata_round4.csv').set_index('sample')
assembly_metadata['pseudo_time_passage'] = assembly_metadata['passage'] 
assembly_metadata.loc[assembly_metadata['exp_type'] =='e003GlycerolRevival','pseudo_time_passage'] = assembly_metadata.loc[assembly_metadata['exp_type'] =='e003GlycerolRevival','passage'] +5
assembly_metadata.head()

In [ ]:
assembly_metadata = pd.read_csv('assembly_metadata_round4.csv').set_index('sample')
assembly_metadata['pseudo_time_passage'] = assembly_metadata['passage'] 
assembly_metadata.loc[assembly_metadata['exp_type'] =='e003GlycerolRevival','pseudo_time_passage'] = assembly_metadata.loc[assembly_metadata['exp_type'] =='e003GlycerolRevival','passage'] +5
assembly_metadata.head()

In [ ]:
fname = '~/git/coalescence-pilot-mgx/workflow/out/midas2_output/old_species/species/metadata.tsv'
df_metadata= pd.read_csv(fname, delimiter = '\t')
df_metadata

def transform_df(df_abundance):
    df_abundance['Lineage'] = df_abundance['species_id'].transform(lambda x: df_metadata.loc[df_metadata['species_id'] == x,'Lineage'].values[0])
    df_abundance['species'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-1])
    df_abundance['genus'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-2])
    df_abundance['family'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[-3])
    df_abundance['phyla'] = df_abundance['Lineage'].transform(lambda x: x.split(';')[1])
    return df_abundance
df_metadata = transform_df(df_metadata).set_index('species_id')
df_metadata 

In [ ]:
folders = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/calculateDiversityDepthv3/*')
all_dfs = []
for folder in folders:
    species_id = folder.split('/')[-1]
    fname1 = f'{folder}/{species_id}_med_depth_df.csv'
    df_depth = pd.read_csv(fname1).set_index('Unnamed: 0').rename(columns={'0': 'depth'})

    fname2 = f'{folder}/{species_id}_num_int_sites2.csv'
    df_num_int= pd.read_csv(fname2).set_index('Unnamed: 0').rename(columns={'0': 'num_int_sites'})

    fname3 = f'{folder}/{species_id}_diversity_df2.csv'
    df_div= pd.read_csv(fname3).set_index('Unnamed: 0').rename(columns={'0': 'diversity'})

    full_df = pd.concat([df_depth, df_num_int, df_div],axis=1)
    full_df['species_id'] = int(species_id)
    full_df['species'] = df_metadata.loc[int(species_id), 'species'].split('s__')[-1]
    
    all_dfs.append(full_df)

    
all_dfs = pd.concat(all_dfs).reset_index().rename(columns = {'Unnamed: 0':'sample'})
all_dfs_assembly = all_dfs.loc[all_dfs['sample'].isin(assembly_metadata.index.values),:]
all_dfs_assembly  = all_dfs_assembly.loc[all_dfs_assembly['depth']>5,:]

#all_dfs_assembly.loc[all_dfs_assembly['sample']=='Assembly-G12-fecal-AA-fecal-0_S703',:]

In [ ]:
all_dfs_assembly 

In [ ]:
all_dfs_assembly = all_dfs.loc[all_dfs['sample'].isin(assembly_metadata.index.values),:]
all_dfs_assembly['species_id'] = all_dfs_assembly['species_id'].astype(int)
all_dfs_assembly['passage'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'passage'])
all_dfs_assembly['pseudo_time_passage'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'pseudo_time_passage'])
all_dfs_assembly['subject'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'subject'])
all_dfs_assembly['media'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'media'])
all_dfs_assembly['type_mesocosm'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'type_mesocosm'])
all_dfs_assembly['mesocosm'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'mesocosm'])
all_dfs_assembly  = all_dfs_assembly.loc[all_dfs_assembly['depth']>=5,:]

In [ ]:
all_dfs_assembly['is_oligo']=all_dfs_assembly['diversity']>1e-3
all_dfs_assembly['is_single']=all_dfs_assembly['diversity']<1e-4
all_dfs_assembly['uncertain']=(all_dfs_assembly['diversity']<1e-3)*(all_dfs_assembly['diversity']>1e-4)
all_dfs_assembly['sp-subject']=all_dfs_assembly['species_id'].astype(str)+'-'+all_dfs_assembly['subject']
all_dfs_assembly_good=all_dfs_assembly.loc[all_dfs_assembly['pseudo_time_passage'].isin([0,5]),:]
all_dfs_assembly_good['is_in']=0
all_dfs_assembly_good['colonization']='unsure'
all_dfs_assembly_good.loc[all_dfs_assembly_good['is_oligo'],'colonization']='oligo'
all_dfs_assembly_good.loc[all_dfs_assembly_good['is_single'],'colonization']='single'
all_dfs_assembly_good.loc[all_dfs_assembly_good['passage']==0,'is_in']=1

In [ ]:
all_dfs_assembly_good=all_dfs_assembly.loc[all_dfs_assembly['subject']!='AC/PP',:]
all_dfs_assembly_good=all_dfs_assembly_good.loc[all_dfs_assembly_good['pseudo_time_passage']==5,:]
all_dfs_assembly_good['log10_int_sites'] = np.log10(all_dfs_assembly_good['num_int_sites']+1)
p=iqplot.strip(all_dfs_assembly_good,q='log10_int_sites',cats=['species'],color_column='subject',q_axis='y',width=900,spread='jitter')
p.xaxis.major_label_orientation=np.pi/2
bokeh.io.show(p)

In [ ]:
fnames = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/calculateFixedDiffsFastv3/*/*_fixed_diffs.csv')
all_dfs = []
for fname in fnames:
    df = pd.read_csv(fname).drop(columns='Unnamed: 0').rename(columns = {'index': 'sample2'})
    good_samples = df.loc[df['sample2'].isin(assembly_metadata.index.values),:]
    good_samples = good_samples.loc[good_samples['sample1'].isin(assembly_metadata.index.values),:]
    species= fname.split('/')[-2]
    good_samples['species'] = df_metadata.loc[int(species), 'species'].split('s__')[-1]
    
  #  good_samples['parents1'] = good_samples['sample1'].transform(lambda x: e003_metadata.loc[e003_metadata['sample']==x,'parent_subjects'].values[0])
   # good_samples['parents2'] = good_samples['sample2'].transform(lambda x: e003_metadata.loc[e003_metadata['sample']==x,
    #                                                         'parent_subjects'].values[0])
    
    all_dfs.append(good_samples)
all_dfs = pd.concat(all_dfs)
good_samples = all_dfs
all_dfs.head()

In [ ]:
good_samples = all_dfs
good_samples['parents1'] = good_samples['sample1'].transform(lambda x: assembly_metadata.loc[x,
                                                             'subject'])
good_samples['parents2'] = good_samples['sample2'].transform(lambda x: assembly_metadata.loc[x,
                                                             'subject'])

good_samples['passage1'] = good_samples['sample1'].transform(lambda x: assembly_metadata.loc[x,
                                                             'passage'])
good_samples['passage2'] = good_samples['sample2'].transform(lambda x: assembly_metadata.loc[x,
                                                             'passage'])

good_samples['media1'] = good_samples['sample1'].transform(lambda x: assembly_metadata.loc[x,
                                                             'media'])
good_samples['media2'] = good_samples['sample2'].transform(lambda x: assembly_metadata.loc[x,
                                                             'media'])

#good_samples=good_samples.loc[(good_samples['passage1']==5)*(good_samples['passage2']==5),:]
#good_samples=good_samples.loc[(good_samples['parents1']!='AC/PP')*(good_samples['parents2']!='AC/PP'),:]
#good_samples=good_samples.loc[good_samples['parents1']!=good_samples['parents2'],:]
good_samples['log_fixed_diffs']=np.log10(good_samples['fixed_diffs']+1)
#good_samples = good_samples.loc[good_samples['num_int_sites']<1e3,:]

In [ ]:
good_samples=good_samples.loc[(good_samples['passage1']==5)*(good_samples['passage2']==5),:]
good_samples = good_samples.loc[(good_samples['parents1']!='AC/PP')*(good_samples['parents2']!='AC/PP'),:]
good_samples['same_subject']=True
good_samples.loc[good_samples['parents1']!=good_samples['parents2'],'same_subject']=False

good_samples['same_media']=True
good_samples.loc[good_samples['media1']!=good_samples['media2'],'same_media']=False

good_samples['diversity']= good_samples['fixed_diffs']/good_samples['comparisons'] # .head()

In [ ]:
good_samples=good_samples.loc[(good_samples['passage1']==5)*(good_samples['passage2']==5),:]
good_samples=good_samples.loc[(good_samples['parents1']!='AC/PP')*(good_samples['parents2']!='AC/PP'),:]

In [ ]:
good_samples['same_subject-same_media']='Diff Subject Diff Media'
good_samples.loc[good_samples['same_subject'],'same_subject-same_media']='Same Subject Diff Media'
good_samples.loc[good_samples['same_media'],'same_subject-same_media']='Diff Subject Same Media'
good_samples.loc[(good_samples['same_media'])*(good_samples['same_subject']),'same_subject-same_media']='Same Subject Same Media'


In [ ]:
good_samples.loc[good_samples['diversity']==0,'diversity']=5e-6

In [ ]:
p2 = hv.Points(good_samples.sort_values(by='same_subject-same_media'),
              kdims = ['same_subject-same_media','diversity'],vdims = [hv.Dimension('diversity',range=(1e-6,1e-1))]).opts(xrotation=60,width=200,
                                                                                           height= 400,
                                                           logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                                        jitter = .3,
                                                                                     #  xaxis=None,
                                                                        cmap = bokeh.palettes.Light[8],  ylim=(5e-6,5e-1),
                                                            xticks=None,
                                                                        show_legend=True,
                                                                        legend_position='right',  
                                                                                     #  multi_level=False,
                                                                  ylabel='Inter Pop Diversity',alpha=.2,xlabel='',           
                                                              color='#00882B')
p2 = hv.render(p2)
p2.yaxis.axis_label_text_font_style = 'normal'
p2.output_backend = 'svg'
export_plot_pdf(p2,'btw_diff')

In [ ]:
len(good_samples.loc[good_samples['num_int_sites'].isna(),:])

In [ ]:
good_samples_diff_sub = good_samples.loc[good_samples['same_subject']==False,:]
good_species = list(np.intersect1d(good_samples_diff_sub['species'].unique(),
                               all_dfs_assembly_good['species'].unique()))
all_dfs_assembly_good['diversity_type']='Wtn'
good_samples_diff_sub['diversity_type']= 'Btw'

p = hv.Points(all_dfs_assembly_good.loc[all_dfs_assembly_good['species'].isin(good_species),:].sort_values(by='species'),
              kdims = ['species','diversity'],vdims = 'diversity').opts(color='black',jitter = .2)
p

v=hv.Violin(good_samples_diff_sub.loc[good_samples_diff_sub['species'].isin(good_species),:].sort_values(by='species'),
            kdims=['species'], vdims=['diversity']).opts(xrotation=60,width=600,inner=None,violin_fill_alpha=.1,
                                                           logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                              violin_color='species',)
v*p

In [ ]:
all_dfs_assembly_good.head()

In [ ]:
good_samples_diff_sub_s1 = good_samples_diff_sub.copy().rename(columns = {'sample1':'sample'})
good_samples_diff_sub_s2 = good_samples_diff_sub.copy().rename(columns = {'sample2':'sample'})
all_dfs_assembly_good['diversity_type'] = 'Wtn'
all_dfs_assembly_good.head()

In [ ]:
all_div = pd.concat([all_dfs_assembly_good[['species','diversity','diversity_type','sample']],
                     good_samples_diff_sub_s1[['species','diversity','diversity_type','sample']],
                     good_samples_diff_sub_s2[['species','diversity','diversity_type','sample']]
                    ])

all_div=all_div.loc[~all_div['diversity'].isna(),:]
all_div['species-sample-compare']=all_div['species']+'-'+ all_div['sample']+'-'+all_div['diversity_type']
all_div['species-sample']=all_div['species']+'-'+ all_div['sample']
all_divWt = all_div.loc[all_div['diversity_type']=='Wtn','species-sample'].unique()
all_divBt = all_div.loc[all_div['diversity_type']=='Btw','species-sample'].unique()
good_compares = np.intersect1d(all_divWt,all_divBt)


In [ ]:
good_species = ['Bacteroides thetaiotaomicron',
 'Bacteroides uniformis',
 'Bacteroides_B dorei',
 'Dorea formicigenerans',
 'Escherichia coli_D',
 'Flavonifractor plautii',
 
 'Veillonella parvula_A']

In [ ]:
all_div['species-dtype']=all_div['species'] + ' '+ all_div['diversity_type']

p = hv.Points(all_div.loc[all_div['species'].isin(good_species),:].sort_values(by='species-dtype'),
              kdims = ['species-dtype','diversity'],vdims = ['diversity','diversity_type']).opts(xrotation=60,width=700,
                                                           logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                                        jitter = .2,
                                                                        cmap = bokeh.palettes.Light[8],
                                                               xlabel='Species',
                                                                        show_legend=True,
                                                                        legend_position='right',       
                                                               ylabel='Diversity',
                                                              color='diversity_type',)
p

In [ ]:
all_div['species-dtype']=all_div['species'] + ' '+ all_div['diversity_type']
all_div_within=all_div.loc[all_div['diversity_type']=='Wtn',:]
all_div_within=all_div_within.loc[~all_div_within['diversity'].isna(),:]
all_div_within = all_div_within.sort_values(by='diversity',ascending=False)
all_div_within['id']=np.arange(len(all_div_within ))
all_div_within['comparison']=all_div_within['species']+'-'+all_div_within['id'].astype(str)
all_div_compare = all_div.loc[all_div['species-sample'].isin(good_compares),:]
p3 = hv
all_div_compare_wtn = all_div_compare.loc[all_div_compare['diversity_type']=='Wtn',:]
all_div_compare_btw = all_div_compare.loc[all_div_compare['diversity_type']=='Btw',:]
p2 = hv.Rectangles([(-100, 1e-3, 500, 1),]).opts(fill_color='#00882B',alpha=.3)
#all_div['species-sample-compare']=all_div['species_id']+'-'+ all_div['sample']+'-'+all_div['diversity_type']
#all_div_within['species-sample']=all_div['species_id'].astype(str)+'-'+ all_div['sample']
all_div_within['id']=np.arange(len(all_div_within)).astype(str)

p = hv.Points(all_div_within.sort_values(by='diversity',ascending=False),
              kdims = ['id','diversity'],vdims = ['diversity','species']).opts(xrotation=60,width=600,
                                                                                           height= 500, size=5,line_color='black',
                                                           logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                                        jitter = .2,
                                                                                       xaxis=None,
                                                                        cmap = bokeh.palettes.Light[8],
                                                               xlabel='Species',xticks=None,
                                                                                           ylim = (5e-6,5e-2),
                                                                        show_legend=True,
                                                                        legend_position='right',  
                                                                                              ylabel='% Polymorphic Sites',
                                                                                     #  multi_level=False,
                                                             #  ylabel='Diversity',
                                                              color='#F39019')


p3=(p2*p).opts(height=200)
#p= hv.render(p)
#p.output_backend = "svg"
#bokeh.io.show(p)
#test_name='bloh'
#bokeh.io.export_svgs (p, filename = test_name + '.svg')
p3 = hv.render(p3)
bokeh.io.show(p3)
p3.output_backend = "svg"
export_plot_pdf(p3,'poly_mutations')

In [ ]:
all_div_within['counts']=1
all_div_within.groupby(['species-sample']).sum(numeric_only=True).sort_values(by='counts',ascending=False)


In [ ]:
p2 = hv.Points(all_div_compare_btw,
              kdims = ['diversity_type','diversity'],vdims = [hv.Dimension('diversity',range=(1e-6,1e-1))]).opts(xrotation=60,width=200,
                                                                                           height= 200,
                                                           logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                                        jitter = .3,
                                                                                       xaxis=None,
                                                                        cmap = bokeh.palettes.Light[8],  ylim=(5e-6,5e-1),
                                                               xlabel='Species',xticks=None,
                                                                        show_legend=True,
                                                                        legend_position='right',  
                                                                                     #  multi_level=False,
                                                                  ylabel='Inter Pop Diversity',alpha=.2,
                                                              color='#00882B')
all_div_compare_btw['log_diversity']=np.log10(all_div_compare_btw['diversity'])
p3 = hv.Violin(all_div_compare_btw,
              kdims = ['diversity_type',],vdims = [hv.Dimension('log_diversity',)]).opts(xrotation=60,width=200,
                                                                                           height= 200,
                                                          # logy=True,
                                                           #logy=True,#ylim=(1,10**5),
                                                                       # jitter = .3,
                                                                                       xaxis=None,violin_fill_color='#00882B',
                                                                       # cmap = bokeh.palettes.Light[8],
                                                               xlabel='Species',xticks=None,
                                                                        show_legend=True,
                                                                        legend_position='right',
                                                                                     ylim=(-5.5,-1.5),
                                                                                     #  multi_level=False,
                                                               ylabel='% Mutations',)
                                                              #color='#00882B')
p3=hv.render(p3)
bokeh.io.show(p3)
p3.output_backend='svg'
export_plot_pdf(p3,'mutations')

In [ ]:
#all_dfs_assembly_good=all_dfs_assembly.loc[all_dfs_assembly['pseudo_time_passage'].isin([0,5]),:]
all_dfs_assembly_good_post=all_dfs_assembly.loc[all_dfs_assembly['subject'].isin(['AA','AE','AF']),:]
all_dfs_assembly_good_post=all_dfs_assembly_good_post.loc[all_dfs_assembly_good_post['pseudo_time_passage'].isin([5]),:]
all_dfs_assembly_good_post

In [ ]:
e003_metadata = pd.read_csv('e003_coalescence_metadata_round4_good.csv').drop(columns = 'Unnamed: 0').set_index('sample')
assembly_metadata = pd.read_csv('assembly_metadata_round4.csv').set_index('sample')

assembly_metadata['pseudo_time_passage']=assembly_metadata['passage']
assembly_metadata.loc[assembly_metadata['exp_type']=='e003GlycerolRevival','pseudo_time_passage']=6
e003_metadata_in = e003_metadata#.loc[e003_metadata['passage']==7,:]
e003_metadata_in_ss = e003_metadata_in.loc[e003_metadata_in['parent_subjects'].isin(['AA-AA','AC/PP-AC/PP',
                                                                                     'AE-AE','AF-AF']),:]

e003_metadata_in_ss['subject'] = e003_metadata_in_ss['parent_subjects'].transform(lambda x: x.split('-')[0])
e003_metadata_in_ss['exp_type'] = 'e003Coal'
e003_metadata_in_ss['pseudo_time_passage'] = e003_metadata_in_ss['passage']+7
e003_metadata_in_ss['comm_og'] = e003_metadata_in_ss['comm']
comm_dic = {
    'AA-mBHI':'B2',
    'AC/PP-mBHI':'B3',
    'AE-mBHI':'B4',
    'AF-mBHI':'B5',
     'AA-mGAM':'B8',
     'AC/PP-mGAM':'B9',
    'AE-mGAM':'B10',
     'AF-mGAM':'B11',
    
}
def get_comm(x):
    subject, _, media, _ = x.split('-')
    meso = f'{subject}-{media}'
    return comm_dic[meso]


e003_metadata_in_ss['comm'] = e003_metadata_in_ss['type_mesocosm'].transform(get_comm)
#print(e003_metadata_in_ss['comm'] )
e003_metadata_in_ss['mesocosm'] = e003_metadata_in_ss['comm'] + '-' + e003_metadata_in_ss['subject']  \
    + '-' + e003_metadata_in_ss['media']
e003_metadata_in_ss['type_mesocosm'] =  e003_metadata_in_ss['subject']  \
    + '-' + e003_metadata_in_ss['media']
#e003_metadata_in_ss = e003_metadata_in_ss.set_index('sample')
assembly_metadata = pd.concat([assembly_metadata, e003_metadata_in_ss[assembly_metadata.columns.values]])
assembly_metadata.head()

In [ ]:
assembly_metadata['exp_type']

In [ ]:
folders = glob('/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/calculateDiversityDepthv3/*')
all_dfs = []
for folder in folders:
    species_id = folder.split('/')[-1]
    fname1 = f'{folder}/{species_id}_med_depth_df.csv'
    df_depth = pd.read_csv(fname1).set_index('Unnamed: 0').rename(columns={'0': 'depth'})

    fname2 = f'{folder}/{species_id}_num_int_sites2.csv'
    df_num_int= pd.read_csv(fname2).set_index('Unnamed: 0').rename(columns={'0': 'num_int_sites'})

    fname3 = f'{folder}/{species_id}_diversity_df2.csv'
    df_div= pd.read_csv(fname3).set_index('Unnamed: 0').rename(columns={'0': 'diversity'})

    full_df = pd.concat([df_depth, df_num_int, df_div],axis=1)
    full_df['species_id'] = int(species_id)
    full_df['species'] = df_metadata.loc[int(species_id), 'species'].split('s__')[-1]
    
    all_dfs.append(full_df)

    
all_dfs = pd.concat(all_dfs).reset_index().rename(columns = {'Unnamed: 0':'sample'})
all_dfs_assembly.columns.values

In [ ]:
all_dfs_assembly = all_dfs.loc[all_dfs['sample'].isin(assembly_metadata.index.values),:]
all_dfs_assembly['species_id'] = all_dfs_assembly['species_id'].astype(int)
all_dfs_assembly['passage'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'passage'])
all_dfs_assembly['pseudo_time_passage'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'pseudo_time_passage'])

all_dfs_assembly['subject'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'subject'])
all_dfs_assembly['media'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'media'])
all_dfs_assembly['type_mesocosm'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'type_mesocosm'])
all_dfs_assembly['mesocosm'] = all_dfs_assembly['sample'].transform(lambda x: assembly_metadata.loc[x,'mesocosm'])
all_dfs_assembly  = all_dfs_assembly.loc[all_dfs_assembly['depth']>=5,:]


In [ ]:
np.sort(all_dfs_assembly['pseudo_time_passage'].unique())

In [ ]:
all_dfs_assembly['species-mesocosm']=all_dfs_assembly['species_id'].astype(str)+'-'+all_dfs_assembly['mesocosm']
all_dfs_assembly= all_dfs_assembly.loc[all_dfs_assembly['subject']!='AC/PP',:]
all_dfs_assembly_14 = all_dfs_assembly.loc[all_dfs_assembly['pseudo_time_passage']==14,:]
all_dfs_assembly_5 = all_dfs_assembly.loc[all_dfs_assembly['pseudo_time_passage']==5,:]
good_stuff = np.intersect1d(all_dfs_assembly_14['species-mesocosm'].unique(),all_dfs_assembly_5['species-mesocosm'].unique())
all_dfs_assembly_14 =all_dfs_assembly_14.loc[all_dfs_assembly_14['species-mesocosm'].isin(good_stuff),:]
all_dfs_assembly_14['div_initial']=np.nan
for spm in good_stuff:
    divs = all_dfs_assembly_5.loc[all_dfs_assembly_5['species-mesocosm']==spm,'diversity'].values[0]
    all_dfs_assembly_14.loc[all_dfs_assembly_14['species-mesocosm']==spm,'div_initial']=divs

In [ ]:
p = hv.Points(all_dfs_assembly_14, vdims= ['diversity',],kdims = ['div_initial','diversity']).opts(logx=True,logy=True,
                                                                                                      color='#F39019',
                                                                                                   xlabel='% Polymorphic Sites P5',
                                                                                                    ylabel='% Polymorphic Sites P14',
                                                                                                   xlim=(5e-6,5e-2),
                                                                                                   width = 300,
                                                                                                   height=250,
                                                                                                   size=5,line_color='black',
                                                                                               ylim=(5e-6,5e-2),)

p2 = hv.Rectangles([(1e-3, 1e-3, 1, 1),]).opts(fill_color='#00882B',alpha=.3,line_color=None,)
p3 = hv.Rectangles([(1e-6, 1e-3, 1e-3, 1),]).opts(fill_color=bokeh.palettes.Pastel2[6][-3],alpha=.5,line_color=None,)
p4 = hv.Rectangles([(1e-3, 1e-6, 1, 1e-3),]).opts(fill_color=bokeh.palettes.Pastel2[6][-4],alpha=.5,line_color=None,)
#p3 = hv.Rectangles([(5e-5, 5e-6, 1e-, 1e-3),]).opts(fill_color='#00882B',alpha=.3,line_color=None,)
p=hv.render(p*p2*p3*p4*p)

bokeh.io.show(p)
p.output_backend='svg'
export_plot_pdf(p, 'div_p5_p14')

In [ ]:
p = hv.Points(all_dfs_assembly_14, vdims= ['diversity',],kdims = ['div_initial','diversity']).opts(logx=True,logy=True,
                                                                                                      color='#F39019',
                                                                                                   xlabel='% Polymorphic Sites P5',
                                                                                                    ylabel='% Polymorphic Sites P14',
                                                                                                   xlim=(5e-6,5e-2),
                                                                                                   width = 300,
                                                                                                   height=250,
                                                                                                   size=5,line_color='black',
                                                                                               ylim=(5e-6,5e-2),)


p=hv.render(p)
p.ray(x=1e-3,y=1e-6,color='grey',alpha=.2,angle=np.pi/2,width=5)
p.ray(x=1e-6,y=1e-3,color='grey',alpha=.2,width=5)
bokeh.io.show(p)
p.output_backend='svg'
export_plot_pdf(p, 'div_p5_p14')

In [ ]:
bokeh.io.show(p)

In [ ]:
all_dfs_assembly['species-mesocosm']=all_dfs_assembly['species_id'].astype(str)+'-'+all_dfs_assembly['mesocosm']
all_dfs_assembly= all_dfs_assembly.loc[all_dfs_assembly['subject']!='AC/PP',:]
all_dfs_assembly_14 = all_dfs_assembly.loc[all_dfs_assembly['pseudo_time_passage']==14,:]
all_dfs_assembly_5 = all_dfs_assembly.loc[all_dfs_assembly['pseudo_time_passage']==5,:]
good_stuff = np.intersect1d(all_dfs_assembly_14['species-mesocosm'].unique(),all_dfs_assembly_5['species-mesocosm'].unique())
all_dfs_assembly_good  =all_dfs_assembly.loc[all_dfs_assembly['species-mesocosm'].isin(good_stuff),:]


In [ ]:
all_dfs_assembly_good['pseudo_time_passage']=all_dfs_assembly_good['pseudo_time_passage'].astype(str)
p = hv.Points(all_dfs_assembly_good.sort_values(by='pseudo_time_passage'), vdims= ['diversity',],kdims = [hv.Dimension('pseudo_time_passage',values = ['1','2','3','4','5','6','7','8',
                                                                                                                                                       '9','10','11','12','13','14',]),
                                                                                                                       'diversity']).opts(logy=True,
                                                                                                      color='#F39019',
                                                                                                   xlabel='Timepoint',
                                                                                                    ylabel='% Polymorphic Sites',ylim=(5e-6,5e-2),
                                                                                                   #xlim=(5e-6,5e-2),
                                                                                                   width = 500,
                                                                                                   height=250,alpha = .5,
                                                                                                             jitter=.2,
                                                                                                   size=5,)
                                                                                               #ylim=(5e-6,5e-2),)


p=hv.render(p)
p.ray(x=1e-3,y=1e-6,color='grey',alpha=.2,angle=np.pi/2,width=5)
p.ray(x=1e-6,y=1e-3,color='grey',alpha=.2,width=5)

p.output_backend='svg'
p.yaxis.axis_label_text_font_style = 'normal'
p.xaxis.axis_label_text_font_style = 'normal'
p.xaxis.axis_label_text_font_size=label_font_size
p.xaxis.major_label_text_font_size=tick_font_size
p.yaxis.axis_label_text_font_size=label_font_size
p.yaxis.major_label_text_font_size=tick_font_size
bokeh.io.show(p)
export_plot_pdf(p, 'div_over_time')